# 1 — k-means in one cohort, described by the demographics

k-means will always hand you k clusters. Nothing in the algorithm says whether
they mean anything, and unlike a dendrogram there is no AU *p*-value to attach —
see the README for why the multiscale bootstrap does not transfer to a flat
partition.

So this notebook does what can be done honestly: pick k in the open, look at how
separated the clusters are, see what distinguishes each one, and check how they
line up against the demographics that were measured independently.

It runs on the SLE proteomics data if you have it, and on a stand-in of the same
shape if you do not.

In [ ]:
import os, shutil, subprocess, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Everything is written here, so nothing lands in the repository.
WORK = Path('sle-run'); WORK.mkdir(exist_ok=True); os.chdir(WORK)

def run(cmd):
    """Run one pvclust-py command and echo it, so the notebook shows the
    command line rather than hiding it behind a function."""
    print('$ ' + ' '.join(cmd if isinstance(cmd, list) else [cmd]))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1500:])
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

def show(png):
    """Render a written figure inline. matplotlib only, so this works in CI too."""
    if not Path(png).exists():
        print(f'(missing {png})'); return
    fig, ax = plt.subplots(figsize=(13, 13))
    ax.imshow(plt.imread(png)); ax.axis('off'); ax.set_title(png, fontsize=9)
    plt.show()

## The data

Download from Zenodo (doi:10.5281/zenodo.20342569), unpack it, and point `SLE` at
the folder.

In [ ]:
# The SLE data is not in the repository -- download it from Zenodo
# (doi:10.5281/zenodo.20342569) and point SLE at the unpacked folder.
SLE = Path(os.environ.get('SLE', '../data/SLE_doi.10.5281_zenodo_20342569'))
REAL = (SLE / 'abundance.csv').exists()

if REAL:
    NBOOT, TOPVAR = 1000, 100
    abundance = pd.read_csv(SLE / 'abundance.csv').set_index('SampleId')
    meta = pd.read_csv(SLE / 'sample-metadata.csv').set_index('SampleId')
    meta = meta[meta['Included_in_study'] == 'Included']
    abundance = abundance.loc[meta.index]
    FEATURE_MAP = str(SLE / 'feature_metadata.txt')
    print(f'SLE data: {abundance.shape[0]} samples x {abundance.shape[1]} reagents')
else:
    # A stand-in with the same shape of problem, so every command below runs
    # unchanged without the download: two batches, a case/control split, and a
    # few proteins measured by more than one reagent.
    NBOOT, TOPVAR = 40, 20
    rng = np.random.default_rng(0)
    n, p = 75, 30
    ids = [f'S{i:03d}' for i in range(n)]
    drivers = rng.normal(size=(n, 6))
    X = np.exp(rng.normal(3, 1, size=(1, p)) + drivers @ rng.normal(size=(6, p))
               + rng.normal(scale=0.3, size=(n, p)))
    seqs = [f'seq.{1000+j}.{j%7}' for j in range(p)]
    abundance = pd.DataFrame(X, index=pd.Index(ids, name='SampleId'), columns=seqs)
    batch = np.where(np.arange(n) % 3 == 0, 'B', 'A')
    abundance.loc[batch == 'B'] *= 1.6                     # a real batch shift
    meta = pd.DataFrame({
        'DonorId': ids, 'Batch': batch,
        'Group': np.where(rng.random(n) < 0.25, 'HV', 'SLE'),
        'Sex': rng.choice(['F', 'M'], n, p=[0.85, 0.15]),
        'Age_group': rng.choice(['26-30', '31-35', '36-40', '41-45'], n),
        'Disease_activity': rng.choice(['Remission', 'LDA', 'MDA', 'HDA'], n),
        'SLEDAI_2K': rng.integers(0, 14, n)}, index=pd.Index(ids, name='SampleId'))
    # names, with three proteins deliberately measured twice
    gene = [f'G{j:02d}' for j in range(p)]
    for a, b in [(1, 2), (10, 11), (20, 21)]:
        gene[b] = gene[a]
    fm = pd.DataFrame({'SeqId': seqs, 'TargetFullName': gene, 'GeneSymbol': gene})
    dup = fm['GeneSymbol'].duplicated(keep=False)
    fm.loc[dup, 'GeneSymbol'] = fm.loc[dup, 'GeneSymbol'] + '_' + fm.loc[dup, 'SeqId']
    fm.to_csv('feature_metadata.txt', sep='\t', index=False)
    FEATURE_MAP = 'feature_metadata.txt'
    print('SLE data not found -- using a stand-in of the same shape.')
    print(f'stand-in: {abundance.shape[0]} samples x {abundance.shape[1]} reagents')

## Three cohorts

Split by donor, not by sample, so a donor with two timepoints lands wholly in one
cohort.

In [ ]:
# Three cohorts, donors kept whole so repeat timepoints never straddle a boundary.
rng = np.random.default_rng(42)
donors = meta.groupby('DonorId').size().index.to_numpy()
who = dict(zip(rng.permutation(donors), range(len(donors))))
which = meta['DonorId'].map(lambda d: 'ABC'[who[d] % 3])

# SLEDAI banded, so it reads as a strip rather than fifteen shades of one colour.
out = meta.copy()
out['SLEDAI_band'] = pd.cut(pd.to_numeric(out['SLEDAI_2K'], errors='coerce'),
                            [-0.1, 0, 4, 8, 30], labels=['0', '1-4', '5-8', '9+'])
out = out.astype({'SLEDAI_band': str}).replace('nan', 'NA').fillna('NA')
out.to_csv('meta.csv')
abundance.to_csv('cohort_all.csv')
for c in 'ABC':
    abundance.loc[which[which == c].index].to_csv(f'cohort{c}.csv')
print({c: int((which == c).sum()) for c in 'ABC'})

## Pick the objects once

This step belongs to `pvclust-py`: selecting features is not specific to either
clustering method, so it exists once rather than twice. Note the order — log2,
then ComBat, then the variance ranking. Ranking before correcting would rank the
batch shift.

In [ ]:
run(['pvclust-py', 'project-features', '--project', 'all',
     '--matrix', 'cohort_all.csv', '--log2',
     '--adjust', 'combat', '--batch-col', 'Batch', '--protect', 'Group',
     '--metadata', 'meta.csv', '--top-variable', str(TOPVAR),
     '--feature-map', FEATURE_MAP, '--feature-label', 'GeneSymbol'])

In [ ]:
# The flags every command shares. Written out in full each time below, so you can
# copy any single cell straight into a terminal.
COMMON = ['--log2', '--adjust', 'combat', '--batch-col', 'Batch',
          '--protect', 'Group', '--metadata', 'meta.csv',
          '--shared-features', 'all_features.csv',
          '--feature-map', FEATURE_MAP, '--feature-label', 'GeneSymbol']
DIST = ['--dist', 'correlation', '--linkage', 'average']
print(' '.join(COMMON))

## Choosing k in the open

No k is *significant*. Silhouette peaks where the clusters are best separated, and
inertia always falls with k, so it only tells you where the fall slows. Read both,
then say which k you used and why.

`--cluster rows` clusters the samples, which is what you want when the clusters
are to be described by demographics.

In [ ]:
run(['kmeans-py', 'choose-k', '--project', 'cohortA', '--k-max', '8',
     '--matrix', 'cohortA.csv', '--cluster', 'rows', *COMMON])

## The partition, and who is in it

**Both axes are clustered, always.** k-means decides the blocks — `--k-rows` on
the samples, `--k-cols` on the objects — and hierarchical clustering orders within
each block and draws it its own small dendrogram. That is the same figure
`ComplexHeatmap` produces from `row_split` and `column_split` with `cluster_rows`
and `cluster_columns` left on.

A block dendrogram is local to its block. k-means is a partition, so the blocks
carry no nesting and no branch lengths between them.

In [ ]:
run(['kmeans-py', 'cluster', '--project', 'cohortA', '--k', '4',
     '--k-rows', '4', '--k-cols', '2',
     '--matrix', 'cohortA.csv', '--cluster', 'rows', *COMMON,
     '--annotate', 'Group,Sex,Age_group,Disease_activity,SLEDAI_band',
     '--dist', 'minkowski', '--linkage', 'ward.D2', '--plot'])

In [ ]:
show('cohortA_heatmap_kmeans.png')

## What each cluster contains

Counts and within-cluster percentages for the categorical variables; median and
interquartile range for the numeric ones, because clinical scores are skewed and a
mean would misdescribe them.

In [ ]:
comp = pd.read_csv('cohortA_composition.csv')
print(comp[comp['variable'] == 'Group'][['cluster', 'level', 'n', 'pct']]
      .to_string(index=False))

## What distinguishes each cluster

A standardised mean difference per cluster and object: the cluster's mean minus
everyone else's, over the pooled standard deviation. Descriptive on purpose, with
no *p*-value — every object was used to build the partition, so testing it against
that same partition would be circular.

**Read this column sceptically.** On the real SLE cohort the smallest cluster came
out defined by five SIGLEC5 and SIGLEC14 reagents sitting a hundred-fold below
everyone else, with tight spread on both sides. That is the SIGLEC14 null
polymorphism — a genotype, not disease biology. k-means will happily hand you a
clean cluster that is a genotype, a batch, or a plate.

In [ ]:
d = pd.read_csv('cohortA_distinguishing.csv')
print(d.groupby('cluster').head(3).to_string(index=False))

---
The associations printed above say what the partition *tracks*, not whether the
clusters are real — the clusters are taken as given. And if a variable drove the
clustering, finding it associated afterwards is circular.

Notebook 2 pools the cohorts without any of them sending a row.